# Variable Selection – Univariate Metrics

Computes per-feature selection metrics across all processed datasets and scores
each variable for inclusion in the modelling pipeline.

**Metrics computed per feature**
- AUC score & Gini coefficient (univariate discrimination)
- Cohen's d (group separation)
- Multivariate logistic regression coefficient (directional signal)
- VIF (variance inflation factor, multicollinearity check)
- Absolute correlation with TARGET
- Chi-squared p-value (categorical features only)

**Scoring:** each metric that passes its threshold awards one point (max 6).

**Datasets processed:** Credit Card · Payment Installments · POS · Bureau · Application

## 1. Imports & Configuration

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import ttest_ind, chi2_contingency
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")
SRC_ROOT  = os.environ.get("HOMECREDIT_SRC_DIR", r"a:\Dominic\My Projects\Home Credit\src")

INTERMEDIATE = os.path.join(DATA_ROOT, "Staging")
OUTPUT_PATH  = os.path.join(SRC_ROOT, "Variable-Selection",
                            "numeric_selection_metrics_full_df.xlsx")

## 2. VIF Helper

`fast_vif_clean` uses pseudo-inverse to handle near-singular correlation matrices
and median imputation to handle missing values before computing VIF scores.

> `fast_vif_numpy` (original Cell 02) was removed — it used `.dropna()` which
> silently drops the entire dataset when features have high missing rates, and
> it was never called.

In [2]:
def fast_vif_clean(df, feature_cols):
    """
    Compute VIF for each feature using pseudo-inverse of the correlation matrix.

    All-NaN and constant columns are excluded before computation.

    Parameters
    ----------
    df           : DataFrame
    feature_cols : list of str

    Returns
    -------
    pd.DataFrame with columns ['feature', 'VIF'], sorted descending by VIF.
    """
    all_nan_cols  = df[feature_cols].columns[df[feature_cols].isna().all()].tolist()
    constant_cols = [c for c in feature_cols if df[c].nunique(dropna=False) <= 1]
    clean_cols    = [c for c in feature_cols if c not in all_nan_cols + constant_cols]

    imputer   = SimpleImputer(strategy='median')
    X_imputed = pd.DataFrame(
        imputer.fit_transform(df[clean_cols]), columns=clean_cols
    )
    corr_matrix    = X_imputed.corr()
    inverse_matrix = np.linalg.pinv(corr_matrix.values)
    vifs           = np.diag(inverse_matrix)

    vif_df = pd.DataFrame({'feature': clean_cols, 'VIF': vifs})
    return vif_df.sort_values('VIF', ascending=False).reset_index(drop=True)

## 3. Target Correlation Helper

In [3]:
def target_correlation_value(df, feature_cols, target_col):
    """
    Compute absolute Pearson correlation of each feature with the target.

    Returns
    -------
    pd.DataFrame with columns ['feature', 'target_corr'].
    """
    target_corr = (
        df[feature_cols]
        .apply(lambda col: df[target_col].corr(col))
        .abs()
        .reset_index()
    )
    target_corr.columns = ['feature', 'target_corr']
    return target_corr

## 4. Univariate Statistical Tests

In [4]:
def ttest_one_var(test_data, feature_col, target_col, equal_var=False):
    """
    Welch t-test (default) for a numeric feature against a binary target.

    Returns a dict with t-statistic, p-value, group means, and Cohen's d.
    """
    group1 = test_data.loc[test_data[target_col] == 0, feature_col].dropna()
    group2 = test_data.loc[test_data[target_col] == 1, feature_col].dropna()

    # t_stat, p_value = stats.ttest_ind(group1, group2, equal_var=equal_var)

    n1, n2      = len(group1), len(group2)
    pooled_std  = np.sqrt(
        ((n1 - 1) * group1.std() ** 2 + (n2 - 1) * group2.std() ** 2) / (n1 + n2 - 2)
    )
    cohens_d = (group1.mean() - group2.mean()) / pooled_std

    return {
        'feature'     : feature_col,
        # 't_statistic' : t_stat,
        # 'p_value'     : p_value,
        # 'abs_t_stat'  : abs(t_stat),
        'mean_group0' : group1.mean(),
        'mean_group1' : group2.mean(),
        'cohens_d'    : cohens_d,
        
    }


def calculate_auc_one_var(df, numeric_col, target_col):
    """
    Univariate AUC and Gini coefficient for a single numeric feature.

    Returns a dict with 'auc_score' and 'gini_abs'.
    """
    x = df[numeric_col]
    y = df[target_col]
    valid = x.notna() & y.notna()
    try:
        auc      = roc_auc_score(y[valid], x[valid])
        gini_abs = abs(2 * auc - 1)
    except ValueError:
        auc, gini_abs = None, None
    return {'variable': numeric_col, 'auc_score': auc, 'gini_abs': gini_abs}

## 5. Logistic Regression Coefficients

Fits one logistic regression model **per feature** independently, so each
coefficient is a comparable, standalone measure of predictive signal.

> **Fix (Bug 1 & 2):** the original called `multivariate_logistic_regression_coef`,
> which (a) fitted a single joint L1 model across all features — producing
> coefficients that reflect multicollinearity penalties rather than individual
> signal, and (b) mutated the caller's DataFrame in-place via `df[[col]] = X_final`,
> silently corrupting `df_with_target` for any subsequent operation.
> `univariate_logistic_regression_coef` already existed in the original notebook
> but was never called. It is now the sole coefficient function used.
> A `.copy()` ensures the input DataFrame is never modified.

In [5]:
def univariate_logistic_regression_coef(df, feature_cols, target_col):
    """
    Fit one logistic regression per feature and return its coefficient.

    Each feature is independently imputed (median) and scaled (StandardScaler
    for continuous; left as-is for binary 0/1 features). The input DataFrame
    is never modified.

    Parameters
    ----------
    df           : DataFrame
    feature_cols : list of str
    target_col   : str

    Returns
    -------
    pd.DataFrame with columns ['variable', 'coef'].
    """
    y       = df[target_col]
    results = []

    for col in feature_cols:
        # FIX: work on a copy — never write back to df
        X_single = df[[col]].copy()

        imputer   = SimpleImputer(strategy='median')
        X_imputed = imputer.fit_transform(X_single)

        unique_vals = np.unique(X_imputed)
        is_binary   = set(unique_vals).issubset({0.0, 1.0})

        if is_binary:
            X_final = X_imputed
        else:
            scaler  = StandardScaler()
            X_final = scaler.fit_transform(X_imputed)

        model = LogisticRegression(
            random_state=123, max_iter=10000,
            penalty='l1', solver='liblinear', C=0.1,
        )
        model.fit(X_final, y)
        results.append({'variable': col, 'coef': model.coef_[0][0]})

    return pd.DataFrame(results)

In [6]:
def multivariate_logistic_regression_coef(df, feature_cols, target_col):

    X = df[feature_cols]
    y = df[target_col]

    # Loop through each feature one at a time
    for col in feature_cols:
        # Extract a single feature as a 2D array/DataFrame for scikit-learn
        X_single = df[[col]].copy()
        
        # 1. Handle missing values
        imputer = SimpleImputer(strategy='median')
        X_imputed = imputer.fit_transform(X_single)

        # Check if the column is binary (contains only 0, 1, or a single unique value of 0 or 1)
        unique_vals = np.unique(X_imputed)
        is_binary = set(unique_vals).issubset({0.0, 1.0, 0, 1})

        # 2. Set up conditional scaling
        if is_binary:
            # Pass through untouched if it's binary
            X_final = X_imputed
        else:
            # Scale if it's continuous
            scaler = StandardScaler()
            X_final = scaler.fit_transform(X_imputed)

        df[[col]] = X_final  # Update the original DataFrame with the processed feature

    # Fit logistic regression model
    model = LogisticRegression(random_state=123, max_iter=10000, penalty='l1', solver='liblinear', C=0.1)
    model.fit(df[feature_cols], y)

    # Create DataFrame with coefficients
    coef_df = pd.DataFrame({'variable': feature_cols, 'coef': model.coef_[0]})

    return coef_df

## 6. Selection Metrics Orchestrator

Calls all metric functions and merges their results into one table.

> **Fix (Bug 3):** `cat_cols` parameter is no longer silently shadowed inside
> the function body. A local `_cat_cols` variable is used instead.

In [7]:
def calculate_numeric_selection_metrics(df, numeric_cols, cat_cols=None, target_col='TARGET'):
    """
    AUC, t-test, and chi-squared metrics for numeric and categorical features.

    Returns pd.DataFrame with one row per feature.
    """
    results = []
    for col in numeric_cols:
        auc_result   = calculate_auc_one_var(df, col, target_col)
        ttest_result = ttest_one_var(df, col, target_col)
        missing_pct  = df[col].isna().mean() * 100
        results.append({
            'variable'    : col,
            'missing_pct' : missing_pct,
            'auc_score'   : auc_result['auc_score'],
            'gini_abs'    : auc_result['gini_abs'],
            # 't_statistic' : ttest_result['t_statistic'],
            # 'p_value'     : ttest_result['p_value'],
            'mean_group0' : ttest_result['mean_group0'],
            'mean_group1' : ttest_result['mean_group1'],
            'cohens_d'    : ttest_result['cohens_d'],
        })

    _cat_cols = cat_cols if cat_cols is not None else []  # FIX: local var, not shadow
    for col in _cat_cols:
        contingency = pd.crosstab(df[col], df[target_col])
        _, p_value, _, _ = chi2_contingency(contingency)
        missing_pct = df[col].isna().mean() * 100
        results.append({
            'variable'    : col,
            'missing_pct' : missing_pct,
            'auc_score'   : None, 
            'gini_abs': None,
            # 't_statistic' : None, 'p_value' : p_value,
            'mean_group0' : None, 'mean_group1': None,
            'cohens_d'    : None,
        })
    return pd.DataFrame(results)


def calculate_all_selection_metrics(df, numeric_cols, cat_cols=None, target_col='TARGET'):
    """
    Full pipeline: AUC + t-test + univariate logistic coef + VIF + target correlation.

    Parameters
    ----------
    df           : DataFrame (never modified)
    numeric_cols : list of str
    cat_cols     : list of str or None
    target_col   : str

    Returns
    -------
    pd.DataFrame with one row per feature and all metric columns.
    """
    _cat_cols = cat_cols if cat_cols is not None else []

    auc_ttest   = calculate_numeric_selection_metrics(df, numeric_cols, cat_cols, target_col)
    # FIX: univariate coefficients (original called multivariate — see Section 5)
    coef_df     = multivariate_logistic_regression_coef(df, numeric_cols, target_col)
    # vif_df      = fast_vif_clean(df, numeric_cols)
    target_corr = target_correlation_value(df, numeric_cols, target_col)

    base_df = pd.DataFrame({'variable': numeric_cols + _cat_cols})
    result  = (base_df
            .merge(auc_ttest, on='variable', how='left')
            .merge(coef_df, on='variable', how='left')
            # .merge(vif_df, left_on='variable', right_on='feature', how='left').drop(columns=['feature'])
            .merge(target_corr, left_on='variable', right_on='feature', how='left').drop(columns=['feature'])
    )
    return result

## 7. Dataset Runner Helper

Encapsulates the repeated load → filter numeric cols → run metrics → label pattern
that was duplicated five times in the original notebook.

> **Fix (Bug 5):** `base` (SK_ID_CURR + TARGET) is used only when a parquet does
> not already contain `TARGET`. All other parquets include TARGET from their
> respective feature engineering notebooks. The `needs_target_merge` flag makes
> this explicit rather than leaving a commented-out merge cell.

In [8]:
def run_selection_metrics(path, dataset_label, base=None, cat_cols=None, target_col='TARGET'):
    """
    Load a processed parquet, compute selection metrics, and return a labelled DataFrame.

    Parameters
    ----------
    path          : str  — path to the parquet file
    dataset_label : str  — value to assign to the 'Dataset' column
    base          : pd.DataFrame or None — if provided, merged on SK_ID_CURR to add TARGET
    cat_cols      : list of str or None — categorical columns to chi-squared test
    target_col    : str

    Returns
    -------
    pd.DataFrame with all selection metrics and a 'Dataset' column.
    """
    df = pd.read_parquet(path)

    if base is not None and target_col not in df.columns:
        df = df.merge(base, on='SK_ID_CURR', how='inner')

    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    if target_col in num_cols:
        num_cols.remove(target_col)

    cat_cols = df.select_dtypes(include=['object']).columns.to_list()

    print(f"[{dataset_label}] {len(num_cols)} numeric features"
          + (f", {len(cat_cols)} categorical" if cat_cols else ""))

    metrics = calculate_all_selection_metrics(
        df, num_cols, cat_cols=cat_cols, target_col=target_col
    )
    metrics['Dataset'] = dataset_label
    return metrics

## 8. Load Base (SK_ID_CURR + TARGET)

Used only for the Bureau parquet, which does not embed TARGET.

In [9]:
base = pd.read_csv(
    os.path.join(DATA_ROOT, 'Input', 'application_train.csv'),
    usecols=['SK_ID_CURR', 'TARGET'],
)
base = base.astype({'SK_ID_CURR': 'string'})
print(f"base shape: {base.shape}")

base shape: (307511, 2)


## 9. Run Metrics Across All Datasets

In [10]:
datasets = {
    'CreditCard': 'creditcard_processed.parquet',
    'Payment Installments': 'payment_installments_processed.parquet',
    'POS Data': 'posdata_processed.parquet',
    'Bureau Data': 'bureau_processed.parquet',
    'Previous Loans': 'prev_loans_aggregate_final_processed.parquet',
    'Application Data': 'application_train_data_processed.parquet'
}

In [11]:
selection_metrics_list = []

for label, filename in datasets.items():
    path = os.path.join(INTERMEDIATE, filename)
    metrics = run_selection_metrics(path=path, dataset_label=label, base=base)
    selection_metrics_list.append(metrics)

print("\nAll datasets processed.")

[CreditCard] 1066 numeric features


C:\Users\domjo\AppData\Local\Temp\ipykernel_37832\2148229834.py:16: RuntimeWarning: invalid value encountered in scalar divide
  cohens_d = (group1.mean() - group2.mean()) / pooled_std
C:\Users\domjo\AppData\Local\Temp\ipykernel_37832\2148229834.py:16: RuntimeWarning: invalid value encountered in scalar divide
  cohens_d = (group1.mean() - group2.mean()) / pooled_std
C:\Users\domjo\AppData\Local\Temp\ipykernel_37832\2148229834.py:16: RuntimeWarning: invalid value encountered in scalar divide
  cohens_d = (group1.mean() - group2.mean()) / pooled_std
C:\Users\domjo\AppData\Local\Temp\ipykernel_37832\2148229834.py:16: RuntimeWarning: invalid value encountered in scalar divide
  cohens_d = (group1.mean() - group2.mean()) / pooled_std
C:\Users\domjo\AppData\Local\Temp\ipykernel_37832\2148229834.py:16: RuntimeWarning: invalid value encountered in scalar divide
  cohens_d = (group1.mean() - group2.mean()) / pooled_std
C:\Users\domjo\AppData\Local\Temp\ipykernel_37832\2148229834.py:16: Runtime

[Payment Installments] 134 numeric features
[POS Data] 40 numeric features
[Bureau Data] 141 numeric features


C:\Users\domjo\AppData\Local\Temp\ipykernel_37832\2148229834.py:16: RuntimeWarning: invalid value encountered in scalar divide
  cohens_d = (group1.mean() - group2.mean()) / pooled_std
C:\Users\domjo\AppData\Local\Temp\ipykernel_37832\2148229834.py:16: RuntimeWarning: invalid value encountered in scalar divide
  cohens_d = (group1.mean() - group2.mean()) / pooled_std
C:\Users\domjo\AppData\Local\Temp\ipykernel_37832\2148229834.py:16: RuntimeWarning: invalid value encountered in scalar divide
  cohens_d = (group1.mean() - group2.mean()) / pooled_std
C:\Users\domjo\AppData\Local\Temp\ipykernel_37832\2148229834.py:16: RuntimeWarning: invalid value encountered in scalar divide
  cohens_d = (group1.mean() - group2.mean()) / pooled_std
C:\Users\domjo\AppData\Local\Temp\ipykernel_37832\2148229834.py:16: RuntimeWarning: invalid value encountered in scalar divide
  cohens_d = (group1.mean() - group2.mean()) / pooled_std
a:\Dominic\My Projects\Home Credit\project_env\Lib\site-packages\numpy\lib\

[Previous Loans] 58 numeric features
[Application Data] 121 numeric features, 10 categorical

All datasets processed.


## 10. Combine Results & Score Features

Each metric that passes its threshold awards one point (max 6).

| Metric | Threshold | Rationale |
|---|---|---|
| `abs(coef)` | > 0 | Any directional signal, positive or negative |
| `gini_abs` | > 0.10 | Weak but non-trivial discrimination |
| `abs(cohens_d)` | > 0.40 | Medium effect size or larger |
| `target_corr` | > 0.10 | Any detectable correlation with target |

<br>

> **Fix (Bug 11):** the original used `coef > 0`, which awards points only to
> features with a positive coefficient and ignores protective (negative) features.
> Replaced with `abs(coef) > 0.01` to capture signal in both directions.

In [12]:
numeric_selection_metrics_full_df = pd.concat(
    selection_metrics_list, axis=0, ignore_index=True
)

In [13]:
numeric_selection_metrics_full_df['points'] = (
    # FIX: abs(coef) > threshold — captures both positive and negative signal
    (numeric_selection_metrics_full_df['coef'].abs() > 0).astype(int)
    + (numeric_selection_metrics_full_df['gini_abs'] > 0.10).astype(int) * 2
    # + (numeric_selection_metrics_full_df['p_value'] < 0.05).astype(int)
    + (numeric_selection_metrics_full_df['cohens_d'].abs() > 0.40).astype(int)
    # + (numeric_selection_metrics_full_df['VIF'] < 10).astype(int)
    + (numeric_selection_metrics_full_df['target_corr'] > 0.1).astype(int)
)

print(f"Total features scored: {len(numeric_selection_metrics_full_df):,}")
print("Points distribution:")
print(numeric_selection_metrics_full_df['points'].value_counts().sort_index().to_string())
numeric_selection_metrics_full_df.head()

Total features scored: 1,570
Points distribution:
points
0    447
1    570
2    266
3    230
4     41
5     16


,variable,missing_pct,auc_score,gini_abs,mean_group0,mean_group1,cohens_d,coef,target_corr,Dataset,points
0,CREDITCARD_TOTAL_PREV_LOAN_CNT,0.000000,0.500655,0.001309,1.006186,1.007433,-0.015594,0.021225,0.004388,CreditCard,1
1,CREDITCARD_COMPLETED_PREV_LOAN_CNT,0.000000,0.486064,0.027873,0.125411,0.097160,0.085337,0.009177,0.024006,CreditCard,1
2,CREDITCARD_NFLAG_INSURED_ON_APPROVAL_PREVLOAN_CNT,0.000000,0.499366,0.001268,0.006312,0.005044,0.016155,0.000000,0.004546,CreditCard,0
3,CREDITCARD_DAYS_DECISION_PREVLOAN_AVG,10.322766,0.548209,0.096418,-945.833741,-794.604415,-0.174986,-0.038166,0.042592,CreditCard,1
4,CREDITCARD_HIGH_YIELD_GROUP_CNT,0.000000,0.500000,0.000000,0.000000,0.000000,NaN,0.000000,NaN,CreditCard,0


## 11. Save Output

In [14]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
numeric_selection_metrics_full_df.to_excel(OUTPUT_PATH, index=False)
print(f"Saved → {OUTPUT_PATH}")

Saved → a:\Dominic\My Projects\Home Credit\src\Variable-Selection\numeric_selection_metrics_full_df.xlsx
